# Importint the libraries

In [ ]:
import MetaTrader5 as mt
from datetime import datetime
import pandas as pd
import numpy as np
import pandas_ta as ta
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.feature_selection import RFE
from sklearn.model_selection import TimeSeriesSplit
from sklearn.impute import SimpleImputer
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning) 
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
#pip install MetaTrader5

In [ ]:
mt.intialize()

In [ ]:
username = int(os.environ['MT5_LOGIN'])
password = os.environ['MT5_PASSWORD']
server = 'Alpari-MT5-Demo'

In [ ]:
mt.login(username, password, server)

In [ ]:
# rates = mt.copy_rates_from('EURUSD', mt.TIMEFRAME_H4,datetime.now(),100)

In [ ]:
ticker = 'EURUSD'
interval = mt.TIMEFRAME_H4
from_date = datetime.now()
no_of_rows = 100
rates = mt.copy_rates_from(ticker, interval, from_date, no_of_rows)
rates

In [ ]:
account_info = mt.account_info()
account_info

In [ ]:
account_info.balance

In [ ]:
account_info.username

In [ ]:
num_symbols = mt.symbols_total()
num_symbols

In [ ]:
symbol_info = mt.symbols_get()
symbol_info

In [ ]:
mt.symbol_info('EURUSD')._asdict()

In [ ]:
import pandas as pd

In [ ]:
ohlc = pd.DataFrame(mt.copy_rates_range('EURUSD', mt.TIMEFRAME_M1, datetime(2023,7,21), datetime.now()))
ohlc

In [ ]:
ohlc['time'] = pd_to_datetime(ohlc['time'], unit='s')
ohlc

In [ ]:
import plotly.express as px
fig = px.line(ohlc, x = ohlc['time'], y = ohlc['close'])
fig.show()

In [ ]:
symbol = 'EURUSD'

request = {
    "action": mt.TRADE_ACTION_DEAL,
    "symbol": symbol,
    "volume": 0.01,
    "type": mt.ORDER_TYPE_BUY, #mt.ORDER_TYPE_SELL
    "price": mt.symbol_info_tick(symbol).ask, #.bid
    "sl": price - 100 * point,
    "tp": price + 100 * point,
    "comment": "python script open",
    "type_time": mt.ORDER_TIME_GTC,
    "type_filling": mt.ORDER_FILLING_IOC,
}
 
# send a trading request
result = mt.order_send(request)

In [ ]:
result

In [ ]:
request = {
    "action": mt.TRADE_ACTION_DEAL,
    "symbol": symbol,
    "volume": 0.01,
    "type": mt.ORDER_TYPE_BUY, #mt.ORDER_TYPE_SELL
    "position": 444129998, # enter the position ID you want to close
    "price": mt.symbol_info_tick(symbol).ask, #.bid
    "sl": price - 100 * point,
    "tp": price + 100 * point,
    "comment": "close the position",
    "type_time": mt.ORDER_TIME_GTC,
    "type_filling": mt.ORDER_FILLING_IOC,
}
order = mt.order_send(request)
order

In [ ]:
# Trading Bot
ticker = 'EURUSD'
qty = 0.01
buy_order_type = mt. ORDER_TYPE_BUY
sell_order_type = me.okoen_tresseLe
buy_price = mt.symbol_info_tick("BTCUSD").ask
sell_price = mt.symbol_info_tick("BTCUSD").bid
s1_pct = 0.05
tp_pct = 0.1
buy_sl = buy_price * (1-s1_pct)
buy_tp = buy_price * (1+tp_pct)
sell_sl = sell_price * (1+s1_pct)
sell_tp = sell_price * (1-tp_pct)

def create_order(ticker, qty, order_type, price, sl, tp):
    request = {
    "action": mt.TRADE_ACTION_DEAL,
    "symbol": ticker,
    "volume": qty,
    "type": order_type,
    "price":price,
    "sl": sl,
    "tp": tp,
    "comment": "python open",
    "type_time": mt.ORDER_TIME_GTC,
    "type_filling": mt.ORDER_FILLING_IOC,
}
    order = mt.order_send(request)
    return order

def close_order(ticker, qty, order_type, price):
    request = {
    "action": mt.TRADE_ACTION_DEAL,
    "symbol": ticker,
    "volume": qty,
    "type": order_type,
    "position": mt.positions_get()[0]._asdict()['ticket'],
    "price": price,
    "comment": "close the position",
    "type_time": mt.ORDER_TIME_GTC,
    "type_filling": mt.ORDER_FILLING_IOC,
}
    order = mt.order_send(request)
    return order

In [ ]:
create_order(ticker, qty, buy_order_type, buy_price, buy_sl, buy_tp)

In [ ]:
mt.positions_get()

In [ ]:
# Pine Script
# LongCondition = close › high[1)
# shortCondition = close ‹ Low[1)
# closelongcondition = close ‹ close [2)
# closeshortcondition = close › close (1)
import time
for i in range(100):
    ohlc = pd.DataFrame(mt.Copy_rates_range('EURUSD', mt. TIMEFRAME_M1, datetime(2023,7,21), datetime.now()))
    ohlc['time']= pd.to_datetime(ohlc['time'], unit='s')
    print(ohlc)

    current_close = list(ohlc[-1:]['close'])[0]
    last_close = list(ohlc[-2:]['close'])[0]
    last_high = list(ohlc[-2:]['high'])[0]
    last_low = list(ohlc[-2:]['low'])[0]

    long_condition = current_close > last_high
    short_condition = current_close < last_low
    closelong_condition = current_close < last_close
    closeshort_condition = current_close > last_close

    already_buy = False 
    already_sell = False

    try:
        already_sell = mt.positions_get()[0]._asdict()['type'] == 1
        already_buy = mt.positions_get()[0]._asdict()['type'] == 0
    except:
        pass

    no_positions= len(mt.positions_get()) == 0

    if long_condition:
        if no_positions:
            create_order(ticker, qty, buy_order_type, buy_price, buy_sl, buy_tp)
            print("Buy Order Placed")
        if already_sell:
            close_order (ticker,qty, buy_order_type, buy_price)
            print('Sell Position Closed')
            time.Sleep(1)
            create_order (ticker, qty,buy_order_type, buy_price, buy_sl, buy_tp)
            print ("Buy Order Placed")
        
    if short_condition:
        if no_positions:
            create_order (ticker, qty, sell_order_type, sell_price, sell_sl, sell_tp)
            print("Sell Order Placed")
        if already_buy:
            close_order (ticker,qty, sell_order_type, sell_price)
            print('buy Position Closed')
            time.Sleep(1)
            create_order (ticker, qty,buy_order_type, sell_price, buy_sl, buy_tp)
            print ("sell Order Placed")

    try:
        already_sell = mt.positions_get()[0]._asdict()['type'] == 1
        already_buy = mt.positions_get()[0]._asdict()['type'] == 0
    except:
        pass

    if closelong_condition and already_buy:
        close_order (ticker,qty,sell_order_type, sell_price)
        print( 'Only Buy Position Closed')

    if closeshort_condition and already_sell:
        close_order(ticker, qty, buy_order_type, buy_price)
        print( 'Only Sell Position Closed' )

    already_buy = False 
    already_sell = False
    time.sleep(60)